# 🤖 STEP 7: LLM Answer Generation

## 🎯 Objective
In this step, we take the **prompt built from retrieved context** and generate an answer using a Large Language Model (LLM).

The pipeline is:



In [22]:
from pathlib import Path
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from gpt4all import GPT4All


In [23]:
VECTOR_STORE = Path(
    r"C:\Users\kgathola.puka\OneDrive - MSC\Documents\GitHub\RCP(test)\SPEED CHATBOT PROJECT\DATA\vector_store"
)

index = faiss.read_index(str(VECTOR_STORE / "faiss.index"))

with open(VECTOR_STORE / "metadata.pkl", "rb") as f:
    chunks = pickle.load(f)

print(f"FAISS vectors: {index.ntotal}")
print(f"Metadata entries: {len(chunks)}")
print("Embedding dimension:", index.d)


FAISS vectors: 178
Metadata entries: 178
Embedding dimension: 768


In [24]:
embedding_model = SentenceTransformer("all-mpnet-base-v2")

In [25]:
def retrieve_context(query, top_k=5, max_chars=1800):
    query_embedding = embedding_model.encode(query, convert_to_numpy=True)

    if query_embedding.ndim == 1:
        query_embedding = query_embedding.reshape(1, -1)

    query_embedding = query_embedding / np.linalg.norm(
        query_embedding, axis=1, keepdims=True
    )

    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        text = chunks[idx]["text"][:max_chars]  # 🔥 HARD LIMIT
        results.append({
            "text": text,
            "source": chunks[idx]["source"],
            "score": float(distances[0][rank])
        })

    return results


In [26]:
def build_prompt(context_chunks, question):
    context = "\n\n".join(
        f"[{c['source']}]\n{c['text']}" for c in context_chunks
    )

    return f"""
**Speed WMS Chatbot** answers questions strictly based on  
Speed WMS documentation using Retrieval-Augmented Generation (RAG).

⚠️ If the information is not found in the documentation, the assistant will say  
**"I do not know."**

Context:
{context}

Question:
{question}

Answer:
""".strip()


In [27]:
llm = GPT4All(
    "mistral-7b-instruct-v0.2.Q5_K_M.gguf",
    device="cpu"
)

In [28]:
def get_llm_answer(prompt):
    return llm.generate(
        prompt,
        max_tokens=250,     # 🔥 keep small
        temp=0.1,           # 🔥 deterministic = faster
        top_p=0.8,
        top_k=40,
        repeat_penalty=1.1,
        streaming=False
    ).strip()


In [30]:
!pip install groq



[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
from groq import Groq
client = Groq(api_key="gsk_REDACTED_KEY_WAS_ROTATED")

def get_llm_answer(prompt):
    completion = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a senior Speed WMS domain expert and trainer.\n"
                    "You must give VERY DETAILED, STEP-BY-STEP answers.\n"
                    "Rules:\n"
                    "- Use ONLY the provided context\n"
                    "- Explain each step clearly\n"
                    "- Use numbered steps and bullet points\n"
                    "- Include sub-steps where relevant\n"
                    "- If something is unclear, let them know that they can contact kgathola Puka for more questions as i'm the devloper or log a ticket"
                    "- If the answer is not in the context, say exactly: I do not know."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,      # factual, stable
        max_tokens=450,       # 🔥 THIS is the main fix
        top_p=0.9
    )

    return completion.choices[0].message.content.strip()


In [32]:
question = "Steps involved in creating receipts?"

print("🔍 Retrieving context...")
context_chunks = retrieve_context(question)

print("🧱 Building prompt...")
prompt = build_prompt(context_chunks, question)

print("🤖 Generating answer...")
answer = get_llm_answer(prompt)

print("\n🧠 BOT ANSWER:\n")
print(answer)

🔍 Retrieving context...
🧱 Building prompt...
🤖 Generating answer...

🧠 BOT ANSWER:

Based on the provided context, the steps involved in creating receipts in Speed WMS are as follows:

**Step 1: Creating a Receipt Header**

1. **Access the contextual menu**:
	* This can be done by clicking on the "Add" button or using a keyboard shortcut (if configured).
	* The contextual menu will appear with various options.
2. **Select "Receipt Header"**:
	* From the contextual menu, select the "Receipt Header" option.
	* This will open a new form for creating a receipt header.
3. **Fill in the required information**:
	* The following elements must be filled in:
		+ Customer account managed activity
		+ Third-party supplier code (receipt selection list)
		+ Receipt reference (alphanumeric type, optional)
4. **Initialize the form**:
	* Some data will be automatically initialized, including:
		+ Speed number reception internal counter (modifiable)
		+ Date and time receipt (default equal to the moment